In [1]:
import dotenv
import openai
import os
import re
import shutil
import json
from pathlib import Path
from glob import glob
from langchain.llms.openai import OpenAI
from langchain.llms import AzureOpenAI
from langchain.embeddings.openai import OpenAIEmbeddings

dotenv.load_dotenv()

# OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
# chatgpt_model_name = os.getenv('OPENAI_MODEL')

azure_deployment_name = os.getenv('AZURE_DEPLOYMENT')
azure_model_name = os.getenv('AZURE_MODEL')

openai.api_type = "azure"
openai.api_key = os.getenv("AZURE_OPENAI_KEY")
openai.api_base = os.getenv('AZURE_ENDPOINT')
openai.api_version = "2023-03-15-preview"

embeddings = OpenAIEmbeddings(
    deployment="test",
    model="text-embedding-ada-002",
    openai_api_type = "azure",
    openai_api_key = os.getenv("AZURE_OPENAI_KEY"),
    openai_api_base = os.getenv('AZURE_ENDPOINT'),
    openai_api_version = "2023-03-15-preview"
)

class NewAzureOpenAI(AzureOpenAI):
    @property
    def _invocation_params(self):
        params = super()._invocation_params
        params.pop('logprobs', None)
        params.pop('best_of', None)
        params.pop('echo', None)
        return params

llm = NewAzureOpenAI(
    openai_api_type = "azure",
    openai_api_key = os.getenv("AZURE_OPENAI_KEY"),
    openai_api_base = os.getenv('AZURE_ENDPOINT'),
    openai_api_version = "2023-03-15-preview",
    model=  os.getenv("AZURE_MODEL"),
    model_kwargs = { "engine": os.getenv("AZURE_DEPLOYMENT") },
)


In [ ]:
# Split audio files
from pydub import AudioSegment

def clean_out_path(fpath):
    out_path = os.path.join(fpath, 'out')

    # Check if the directory exists
    if os.path.exists(out_path):
        # Cleanup: delete all files in the directory
        for filename in os.listdir(out_path):
            file_path = os.path.join(out_path, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print('Failed to delete %s. Reason: %s' % (file_path, e))
    else:
        # Directory does not exist, so create it
        os.makedirs(out_path)

def split(audio, filename):
    ten_minutes = 20 * 60 * 1000
    pos = 0
    idx = 1
    while pos < len(audio):
        print("Saving {0}".format(filename.format(idx)))
        chunk = audio[pos:min(pos + ten_minutes, len(audio) - 1)]
        chunk.export(filename.format(idx), format="mp3")
        idx += 1
        pos += ten_minutes

for fdir in ["2017", "2018"]:
    outdir = os.path.join(os.path.curdir, fdir)
    for fname in glob("{0}/*.mp3".format(fdir)):
        print("Splitting {0}".format(fname))
        audio = AudioSegment.from_mp3(fname)
        outname = Path(os.path.splitext(fname)[0]).stem + " part {0}.mp3"
        split(audio, os.path.join(outdir, "out", outname))

In [ ]:
# Transcribe
for fdir in ["2017", "2018"]:
    outdir = os.path.join(os.path.curdir, fdir, "out")
    n = 1
    while n > 0:
        files = glob("{0}/*.mp3".format(outdir))
        n = 0
        for fname in files:
            outname = re.sub(r"\.mp3$", ".srt", fname)
            if not os.path.exists(outname):
                with open(fname, "rb") as audio_file:
                    print("transcribing {0}".format(fname))
                    transcript = openai.Audio.transcribe("whisper-1", audio_file, response_format="srt", api_key=OPENAI_API_KEY)
                    with open(outname, "wt") as srt_file:
                        srt_file.write(transcript)
                n += 1

In [ ]:
#Splitting text files (questions)
def save_chunk(prefix, chunk, file_count):
    with open(f'{prefix}{file_count:03d}.txt', 'w') as f:
        f.write(''.join(chunk))

def process_file(filename):
    size = 0
    chunk = []
    file_count = 1
    regex = re.compile(r'^[0-9]+\.')

    with open(filename, 'rt') as f:
        for line in f:
            if regex.match(line):
                save_chunk(filename, chunk, file_count)
                chunk = []
                size = 0
                file_count += 1

            size += len(line)
            chunk.append(line)

    # Save the last chunk if it's not empty
    if chunk:
        save_chunk(filename, chunk, file_count)

for f in glob("text/*.txt"):
    process_file(f)

In [2]:
# Test Chat
messages=[
    {"role": "user", "content": "What is azure cognitive services?"}
]
print(openai.ChatCompletion.create(
        engine="test-32k",
        model="gpt4-32k",
        messages=messages)["choices"][0]["message"]["content"])

# print(openai.ChatCompletion.create(
#         engine="test2",
#         model="gpt-3.5-turbo",
#         messages=messages))


Azure Cognitive Services is a suite of AI-powered services and APIs offered by Microsoft, designed to help developers build intelligent applications that utilize natural language understanding, computer vision, speech recognition, and other advanced machine learning capabilities. These services enable applications to recognize and interpret human behaviors, such as spoken language, facial expressions, emotions, gestures, or handwriting. This makes it easier for developers to add AI capabilities to their applications without having extensive expertise in machine learning and artificial intelligence. Azure Cognitive Services include tools for language understanding, speech translation, text translation, computer vision, and more.


In [ ]:
# Text files to OpenAI to clean JSON database
def f(text):
    messages=[
        {"role": "system", "content": "You are an assistant.This text contains one or two exam questions. Parse and produce a JSON array containing questions. each item should contain fields {enunciate, answer, explanation} extracted from the text. Each item has a question prompt, an answer and an explanation. Answer and explanation may be the same. Rewrite the explanation the best that you can, provided the original explanation. Enunciate is the transcription closest to original text of question prompt, fixed syntax and meaning. If there are more than one question, repeat the enunciate in each item, otherwise it is null. Output must be a JSON string parseable by python. Rewrite the text to fix syntax without changing the meaning."},
        {"role": "user", "content": text}
    ]
    # TO CHECK: I may have confused engine and model up in the beginning
    return openai.ChatCompletion.create(
            engine=azure_engine_name,
            model=azure_model_name,
            messages=messages)["choices"][0]["message"]["content"]

def process_file():
    with open('text/trimmed.txt', 'r') as file, open('text/processed.txt', 'w') as processed, open('text/output.json', 'w') as json_output:
        content = file.read()
        chunks = content.split('\n\n---\n\n')
        clean = []
        json_array = []

        c = 0
        for chunk in chunks:
            if chunk:
                print(".", end="")
                processed.write(chunk + '\n\n---\n\n')
                try:
                    clean = clean + [f(re.sub(r'[^\x20-\x7E]', '', chunk))]
                    processed.write(clean[-1])
                    try:
                        json_array = json_array + json.loads(clean[-1].replace('\\n', '\n').replace('\\\\', '\\'))
                    except:
                        processed.write('\nERROR PARSING JSON\n\n***\n\n')
                    processed.write('\n\n***\n\n')
                except:
                    processed.write("ERROR\n\n***\n\n")
        json.dump(json_array, json_output, indent=4)
        return clean

clean = process_file()

In [ ]:
# get_embeddings
def get_embeddings(text):
   if text:
      return embeddings.embed_query(text)
   else:
      return None

get_embeddings("This is a test")

In [ ]:
# Read from JSON, save embeddings to CSV
from langchain.document_loaders import TextLoader
from langchain.document_loaders import DirectoryLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores.pgvector import PGVector, DistanceStrategy
from langchain.docstore.document import Document

def format_response(enunciate, answer, explanation):
  if answer and explanation:
    if answer.upper() == "TRUE":
      return enunciate + "\n\nTrue. " + explanation
    elif answer.upper() == "FALSE":
      return enunciate + "\n\nFalse. " + explanation
    else:
      return enunciate + "\n\n" + answer + ". " + explanation
  elif answer:
    return enunciate + "\n\n" + answer
  elif explanation:
    return enunciate + "\n\n" + explanation
  else:
    return None

data = json.load(open('text/output.json', 'r'))
docs = []
print("Loading documents")
i = 0
for item in data:
    i += 1
    if i > 343:
      print("{0}/{1}".format(i, len(data)), end="\r")
      u = { "enunciate": item["enunciate"], \
            "answer": item["answer"], \
            "explanation": item["explanation"], \
            "combined" : format_response(item["enunciate"], item["answer"], item["explanation"]),
            "embedding_enunciate": get_embeddings(item["enunciate"]),\
            "embedding_answer": get_embeddings(item["answer"]),\
            "embedding_explanation": get_embeddings(item["explanation"]),\
            "embedding_combined": get_embeddings(format_response(item["enunciate"], item["answer"], item["explanation"]))}
      docs.append(u)
print("Done loading documents")
with open('json_embedded.json', 'wt') as f:
    f.write(json.dumps(docs, indent=2))

In [ ]:
# Create database
import psycopg
from psycopg.conninfo import make_conninfo

pg_string = make_conninfo(
  host=os.getenv('DB_HOST'),
  port=os.getenv('DB_PORT'),
  dbname=os.getenv('DB_DATABASE'),
  user=os.getenv('DB_USER'),
  password=os.getenv('DB_PASSWORD')
)


table_sql = """
drop table docs;
create table docs (
  enunciate text not null,
  answer text,
  explanation text,
  combined text not null,
  embedding_enunciate vector(1536),
  embedding_answer vector(1536),
  embedding_explanation vector(1536),
  embedding_combined vector(1536)
);
ALTER TABLE docs ADD CONSTRAINT unique_enunciate UNIQUE (enunciate);
"""

with psycopg.connect(pg_string) as conn:
    conn.execute(table_sql)


In [ ]:
# From JSON to database
with psycopg.connect(pg_string) as conn:
    docs = json.loads(open('json_embedded.json', 'r').read())
    sql = "INSERT INTO docs (enunciate, answer, explanation, combined, embedding_enunciate, embedding_answer, embedding_explanation, embedding_combined) VALUES (%s, %s, %s, %s, %s, %s, %s, %s) ON CONFLICT (enunciate) DO NOTHING;"

    for doc in docs:
        if doc["combined"]:
            conn.execute(sql, (doc["enunciate"], doc["answer"], doc["explanation"], doc["combined"], doc["embedding_enunciate"], doc["embedding_answer"], doc["embedding_explanation"], doc["embedding_combined"]))
        else:
            print(doc)
        conn.commit()

In [ ]:
# Query database
query = get_embeddings("Cornout collusion")

with psycopg.connect(pg_string) as conn:
    results = conn.execute("SELECT * FROM docs ORDER BY embedding_combined <=> %s::vector LIMIT 5;", [query])
    out = results.fetchall()

titles = [row[0] for row in out]
print("\n".join(titles))

In [ ]:
# From text documents to database

from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores.pgvector import PGVector, DistanceStrategy
from langchain.docstore.document import Document

conn_string = "postgresql://{0}:{1}@{2}:{3}/{4}".format(os.getenv('DB_USER'), os.getenv('DB_PASSWORD'), os.getenv('DB_HOST'), os.getenv('DB_PORT'), os.getenv('DB_DATABASE'))
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100, separator=" ")

lectures_docs = text_splitter.split_documents(lectures)
db_lectures = PGVector.from_documents(embedding=embeddings, documents=lectures_docs, collection_name="lectures", connection_string=conn_string, distance_strategy=DistanceStrategy.COSINE)

books_docs = text_splitter.split_documents(books)
db_books = PGVector.from_documents(embedding=embeddings, documents=books_docs, collection_name="books", connection_string=conn_string, distance_strategy=DistanceStrategy.COSINE)

In [ ]:
# Test database

[c.page_content for c,r in db_lectures.similarity_search_with_score("Cornout collusion", 10)]
[c.page_content for c,r in db_books.similarity_search_with_score("Cornout collusion", 10)]

In [ ]:
import psycopg
from psycopg.conninfo import make_conninfo
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores.pgvector import PGVector, DistanceStrategy
from langchain.docstore.document import Document

pg_string = make_conninfo(
  host=os.getenv('DB_HOST'),
  port=os.getenv('DB_PORT'),
  dbname=os.getenv('DB_DATABASE'),
  user=os.getenv('DB_USER'),
  password=os.getenv('DB_PASSWORD')
)

conn_string = "postgresql://{0}:{1}@{2}:{3}/{4}".format(os.getenv('DB_USER'), os.getenv('DB_PASSWORD'), os.getenv('DB_HOST'), os.getenv('DB_PORT'), os.getenv('DB_DATABASE'))

db_lectures = PGVector(embedding_function=embeddings,collection_name="lectures", connection_string=conn_string, distance_strategy=DistanceStrategy.COSINE)
db_books = PGVector(embedding_function=embeddings, collection_name="books", connection_string=conn_string, distance_strategy=DistanceStrategy.COSINE)

In [ ]:
with psycopg.connect(pg_string) as conn:
    docs = json.loads(open('json_embedded.json', 'r').read())
    sql = "INSERT INTO docs (enunciate, answer, explanation, combined, embedding_enunciate, embedding_answer, embedding_explanation, embedding_combined) VALUES (%s, %s, %s, %s, %s, %s, %s, %s) ON CONFLICT (enunciate) DO NOTHING;"

    for doc in docs:
        if doc["combined"]:
            conn.execute(sql, (doc["enunciate"], doc["answer"], doc["explanation"], doc["combined"], doc["embedding_enunciate"], doc["embedding_answer"], doc["embedding_explanation"], doc["embedding_combined"]))
        else:
            print(doc)
        conn.commit()

In [ ]:
# NOT Working for GPT-4
from langchain.chains import RetrievalQA

retriever = db_books.as_retriever(search_type="similarity", search_kwargs={"k":2})
qa = RetrievalQA.from_chain_type(llm,chain_type="stuff", retriever=retriever)
query = "Cornout collusion?"
result = qa({"query": query})
print(result)

In [ ]:
from langchain.chains.question_answering import load_qa_chain
chain = load_qa_chain(llm, chain_type="stuff")
chain.run(input_documents=docs, question=query)

In [ ]:
# u = db_books.similarity_search("Cornout collusion", 10)
# t = db_lectures.similarity_search("Cornout collusion", 10)
u = db_books.similarity_search("Suppose the government replaces a firm-by-firm air pollution quota with a tax that keeps total air pollution constant. True, False, and Explain: This tax will almost certainly be a Kaldor-Hicks improvement, but almost certainty not be a Pareto improvement", 10)
t = db_lectures.similarity_search("Suppose the government replaces a firm-by-firm air pollution quota with a tax that keeps total air pollution constant. True, False, and Explain: This tax will almost certainly be a Kaldor-Hicks improvement, but almost certainty not be a Pareto improvement", 10)
print(sum([len(i.page_content) for i in u[:5]]))
print(sum([len(i.page_content) for i in t[:5]]))

In [ ]:
# Query & Answer implemented
def query(q):
    # Test Chat
    u = db_books.similarity_search(q, 10)
    t = db_lectures.similarity_search(q, 10)
    idx_u = 0
    tt = 0
    while idx_u < len(u):
        tt += len(u[idx_u].page_content)
        if tt < 8000:
            idx_u += 1
        else:
            break

    tt = 0
    idx_t = 0
    while idx_t < len(t):
        tt += len(u[idx_t].page_content)
        if tt < 8000:
            idx_t += 1
        else:
            break

    print("u={0} t={1}".format(idx_u, idx_t))

    messages=[
        {"role": "system", "content": "You are an assistant. Answer concisely using input data."},
        {"role": "system", "content": "Input data are: " + "\n".join([i.page_content for i in u[:idx_u]]) + "\n".join([i.page_content for i in t[:idx_t]])},
        {"role": "user", "content": q}
    ]
    # TO CHECK: I may have confused engine and model up in the beginning
    print(openai.ChatCompletion.create(
            engine="test-32k",
            model="gpt4-32k",
            messages=messages))

query("Suppose the government replaces a firm-by-firm air pollution quota with a tax that keeps total air pollution constant. True, False, and Explain: This tax will almost certainly be a Kaldor-Hicks improvement, but almost certainty not be a Pareto improvement")

In [7]:
import time
from langchain.text_splitter import CharacterTextSplitter

def run_chat(message, level):
    print("run chat")
    if level == 1:
        messages=[
            {"role": "system", "content": "You are an assistant. The user will tell you a transcript for a meeting. Write a summary of the meeting with as much detail as possible in a consice narration of the topics discussed. Then list the highlights."},
            {"role": "user", "content": message}
        ]
    else:
        messages=[
            {"role": "system", "content": "The user will tell you summary and highlights for a meeting. Summarize summaries and consolidate highlights."},
            {"role": "user", "content": "Summarize the text below.\n\n" + message}
        ]
    time.sleep(5)
    reply = openai.ChatCompletion.create(
            engine="test-32k",
            model="gpt-4",
            messages=messages)["choices"][0]["message"]["content"]
    # reply = openai.ChatCompletion.create(
    #         engine="test2",
    #         model="gpt-3.5-turbo",
    #         messages=messages)["choices"][0]["message"]["content"]
    return reply

def run_summarizer(q, level = 1):
    idx_start = 0
    summary = ""
    for i in range(len(q)):
        if sum([len(u) for u in q[idx_start:i]]) > 40000:
            reply = run_chat("\n".join(q[idx_start:i]), level)
            summary = summary + reply
            idx_start = i
    summary = summary + run_chat("\n".join(q[idx_start:]), level)
    return summary

def summarize(text, length = 1500):
    level = 1
    while len(text) > length:
        text_splitter = CharacterTextSplitter(chunk_size=4000, chunk_overlap=0)
        texts = text_splitter.split_text(text)
        text = run_summarizer(texts, level)
        level += 1
        print(text)
    return text

print(summarize(open('/mnt/deo-mini/drop/meeting_2.txt', 'r').read()))

# In the meeting, the team discussed limitations in the Bring Your Own Data project, specifically the inability to attach more than one Cognitive Search Index and the automatic chunking process. They considered using multiple indexes for different data sources and possibly trying out vector stores for more accurate responses. The team also talked about booking a custom engagement with Microsoft to help fine-tune their solution, review architecture, and address security concerns. They plan to try using Pinecone or the new vector store with Lang Chain before enlisting Microsoft's assistance.

# The meeting focused on discussing limitations and best practices for the "Bring Your Own Data" project. It was suggested to create an index for each data source and use vector stores instead of COG search. A "proof of concept" and custom engagement options were also discussed. The meeting also included discussions on engagement options for building a lifecycle solution and the need for security review. Additionally, concerns were raised about the limitations of the prebuilt chat with data app and the need for an architecture review. The team plans to complete the POC by the end of July and explore the possibility of an internal beta. Finally, there will be a follow-up session with the product owner about the vector store.



run chat
Summary:

In the meeting, the team discussed methods to make the ICS (International Comparison Program) work with I data while maintaining compatibility with ECOS. They proposed three potential solutions to achieve this, with a focus on a short-term solution while also considering a long-term strategy. The first option involved keeping ECOS dependency for incoming data while allowing ICS to consume data from both I data and ECOS. The second option required ICS to make code changes to read and write data from both I data and ECOS. The third option involved changing ICS to use the I data naming convention for time series and rebuild the report forms.

After a thorough discussion, the team agreed on a tactical solution, which involves building a bridge to map between the old time series codes and the new I data codes. This solution would enable ICS to read and write data to both ECOS and I data for different products, such as CPI and BOP. This approach requires fixing the issue w

In [ ]:
from langchain.chains.summarize import load_summarize_chain
from langchain.docstore.document import Document
from langchain.text_splitter import CharacterTextSplitter

text = open('/mnt/deo-mini/drop/meeting.txt', 'r').read()
text_splitter = CharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
texts = text_splitter.split_text(text)
docs = [Document(page_content=t) for t in texts[:3]]

chain = load_summarize_chain(llm, chain_type="map_reduce")
u = chain.run(docs)

print(u)

#The group discusses the indexing of multiple data sources when using Elasticsearch. They agree that having multiple indexes is manageable. Tirzah VanDamme advises Ina Darsadze to start with the vector store to resolve the accuracy issues with the responses. She also suggests that Darsadze integrates the repo given to her to her chat PDF test environment. Jose Deodoro mentions the impressive results of the vector search over the weekend and asks VanDamme if she suggests reusing Azure Pinecone or SQL or POSTGRES.<|im_end|>